**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/hazon-2022"], check=True)
    os.chdir("/content/encore/entries/hazon-2022")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Hazon et al. (2022) from CRCNS hc-30: IMSE fits with the paper's repetitions

**Paper.** Hazon O, Minces VH, Tomàs DP, Ganguli S, Schnitzer MJ, Jercog PE (2022). Noise
correlations in neural ensemble activity limit the accuracy of hippocampal spatial
representations. *Nature Communications* 13, 4276.
[doi:10.1038/s41467-022-31254-y](https://doi.org/10.1038/s41467-022-31254-y)

**Dataset.** CRCNS hc-30, [doi:10.6080/K0GH9G5X](https://doi.org/10.6080/K0GH9G5X), file
`hc-30/live/event_traces.mat` streamed from the public S3 bucket and checked against its MD5
(version `unversioned`, as in the other two notebooks, whose headers give the details).

**Why this notebook.** The decoding notebook fits IMSE(n) = I0 n / (1 + n/N) to 4 neuron subsets
per ensemble size on sizes 1, 25, 50, ... (to fit the time limit), where the paper uses 80
subsets on sizes 1, 10, 20, ... (Fig. 2e legend, "an average of over 80 randomly chosen subsets
of cells"; code decode_series, d_neurons 10). The verifier of round 1 asked whether the claims
built on the fitted N and I0 (saturation-ensemble-size, suppl11b-i0-vs-single-cell-snr,
suppl8-task-performance-and-accuracy) fail because of the reduced repetitions. Running 80
subsets on the paper's grid for all 73 large-data sessions would take about 4 hours here, so
this notebook runs it on a set of sessions chosen before any fit was seen:

- every large-data session of Mouse2011 and Mouse2012, the two animals with the lowest per-animal
  N in the decoding notebook (they set the lower bound of the saturation range, and Mouse2011's
  mean I0 decides suppl11b);
- the large-data session of Mouse2028, the animal with the highest per-animal N, with the
  fewest trials (the cost of a decoder grows with the square of the trial count; its other 14
  sessions would take 10 to 15 minutes each on one process).

For each session it reproduces the decoding notebook's fit (same seeds and settings, so the same
numbers) and fits again to 80 subsets on 1, 10, 20, ... and all neurons, unshuffled only (the
three claims use the unshuffled fit). The claims are tested in the decoding notebook; this one
shows how the fit depends on the number of subsets.

Added in the second revision, at the round-2 verifier's request. The whole entry has to run
within two hours here, and the first two notebooks and the part above take about 105 minutes,
so the additions are small:

- Mouse2023, the animal that sets the upper bound of the decoding notebook's variant without
  misaligned sessions and degenerate fits (its one large-data session), fitted again with 40
  subsets on steps of 10. Mouse2022, which sets the lower bound, is not repeated: its five
  sessions have 93 to 185 trials per direction, and 80 subsets on steps of 10 would take about
  5 CPU hours here.
- How N depends on the decoder: the 4-subset run repeated with C = 0.1, C = 10 and MATLAB's
  default loss-weighted decoding (hinge loss, C = 1) in place of voting, on 9 sessions whose
  events track position: the three of Mouse2012, the five of Mouse2011 with 250 neurons or
  fewer, and Mouse2023's.

Run time about 45 minutes on 4 CPUs.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_v] = "1"  # one thread per process; the decoders run in 4 processes
import multiprocessing as mp
import resource
import time

import numpy as np
import pandas as pd

import hc30

T0 = time.time()
sessions, md5, load_s = hc30.load_sessions()
print(f"streamed and parsed event_traces.mat in {load_s:.0f} s; md5 {md5} "
      f"({'matches' if md5 == hc30.MD5 else 'DIFFERS FROM'} checksums.md5)")

# The decoding notebook's settings, reproduced here
REPS_FULL = 10
REPS_SUB = 4
SIZES = [1] + list(range(25, 500, 25))
C_SVM = 1.0
# The paper's settings
REPS_PAPER = 80   # Fig. 2e legend
STEP_PAPER = 10   # code decode_series, default_opt d_neurons 10
# Added in the second revision (see the header)
REPS_2023 = 40   # Mouse2023
VARIANTS = [("C = 0.1", 0.1, "vote"), ("C = 10", 10.0, "vote"), ("loss-weighted", 1.0, "lossweighted")]

## Data check and choice of sessions

In [ ]:
rows, TENS = [], {}
for i, s in enumerate(sessions):
    T, d, tr = hc30.session_tensor(s)
    rows.append(dict(i=i, mouse=s["mouse"], session=s["session_id"], neurons=T.shape[0],
                     right=int((d == 1).sum()), left=int((d == -1).sum())))
    TENS[i] = (T, d)
info = pd.DataFrame(rows)
info["min_trials"] = info[["right", "left"]].min(axis=1)
info["large"] = (info.neurons > 200) & (info.min_trials > 30)
print(f"sessions {len(info)}, mice {info.mouse.nunique()}, large-data sessions {int(info.large.sum())}")
low = info[info.large & info.mouse.isin(["Mouse2011", "Mouse2012"])]
high = info[info.large & (info.mouse == "Mouse2028")].nsmallest(1, "min_trials")
CHOSEN = list(low.i) + list(high.i)
EXTRA = list(info[info.large & (info.mouse == "Mouse2023")].i)
sel = info.loc[CHOSEN + EXTRA].copy()
sel["clock_err_cm"] = [hc30.clock_check(sessions[i], 0) for i in CHOSEN + EXTRA]
sel["misaligned"] = sel.clock_err_cm > 12  # as in the other notebooks
print(f"{len(CHOSEN)} sessions chosen, then {len(EXTRA)} of Mouse2023:")
print(sel[["i", "mouse", "session", "neurons", "right", "left", "clock_err_cm", "misaligned"]]
      .round(1).to_string(index=False))
TENS = {i: TENS[i] for i in CHOSEN + EXTRA}
VAR_SESSIONS = [i for i in CHOSEN + EXTRA
                if info.mouse[i] in ("Mouse2011", "Mouse2012", "Mouse2023") and not sel.misaligned[i]
                and (info.mouse[i] != "Mouse2011" or info.neurons[i] <= 250)]
print("sessions for the decoder variants:", VAR_SESSIONS)

In [ ]:
# Cost estimate: 2 decoders at 100 neurons per session, times the number of decoders
def plan(i):
    """Subsets and step of the second fit: the paper's, or fewer for Mouse2023 (see the header)."""
    if info.mouse[i] == "Mouse2023":
        return REPS_2023, STEP_PAPER
    return REPS_PAPER, STEP_PAPER

t_est, PER = 0, {}
rng = np.random.default_rng(0)
for i in CHOSEN + EXTRA:
    T, d = TENS[i]
    t = time.time()
    for _ in range(2):
        idx = hc30.equalize_trials(d, rng)
        sub = rng.choice(T.shape[0], 100, replace=False)
        hc30.decode_once(T[sub][:, :, idx], d[idx], rng, False, C=C_SVM)
    PER[i] = (time.time() - t) / 2
    reps, step = plan(i)
    n_sizes = len(hc30.ensemble_sizes(T.shape[0], step)) + 1
    t_est += PER[i] * (n_sizes * reps + 2 * REPS_FULL + 2 * REPS_SUB * len(SIZES))
    if i in VAR_SESSIONS:
        t_est += PER[i] * len(VARIANTS) * (REPS_FULL + REPS_SUB * len(SIZES))
print(f"estimated {t_est / 60:.0f} min on one process, about {t_est / 240:.0f} min on 4")

## Decoding runs

In [ ]:
def job(i):
    T, d = TENS[i]
    n = T.shape[0]
    # the decoding notebook's run, same seeds
    full = hc30.decode_full(T, d, seed=20000 + i, reps=REPS_FULL, C=C_SVM)
    sizes4 = [s for s in SIZES if s < n]
    iu4, _ = hc30.decode_series(T, d, seed=30000 + i, sizes=sizes4, reps=REPS_SUB, C=C_SVM)
    sizes4 = np.array(sizes4 + [n])
    imse4 = np.r_[iu4, np.mean(1 / np.array(full["mse_u"]))]
    # the paper's repetitions and grid (80 subsets at 1, 10, 20, ..., all neurons), or fewer
    reps, step = plan(i)
    sizes80 = np.array([1] + hc30.ensemble_sizes(n, step))
    imse80, _ = hc30.decode_series(T, d, seed=60000 + i, sizes=sizes80, reps=reps, C=C_SVM,
                                   shuffled=False)
    return i, sizes4, imse4, sizes80, imse80


def variant_job(i, k):
    """The decoding notebook's 4-subset run, unshuffled, with another decoder setting."""
    _, C, rule = VARIANTS[k]
    T, d = TENS[i]
    n = T.shape[0]
    sizes4 = [s for s in SIZES if s < n]
    iu4, _ = hc30.decode_series(T, d, seed=70000 + 1000 * k + i, sizes=sizes4, reps=REPS_SUB, C=C,
                                shuffled=False, decoding=rule)
    ifull, _ = hc30.decode_series(T, d, seed=75000 + 1000 * k + i, sizes=[n], reps=REPS_FULL, C=C,
                                  shuffled=False, decoding=rule)
    return i, k, np.array(sizes4 + [n]), np.r_[iu4, ifull]


def any_job(a):
    return ("main", job(a[1])) if a[0] == "main" else ("var", variant_job(a[1], a[2]))


def cost(i):
    reps, step = plan(i)
    return PER[i] * reps * len(hc30.ensemble_sizes(info.neurons[i], step))


tasks = [("main", i) for i in sorted(CHOSEN + EXTRA, key=lambda i: -cost(i))]
tasks += [("var", i, k) for i in VAR_SESSIONS for k in range(len(VARIANTS))]
t = time.time()
with mp.get_context("fork").Pool(4) as pool:
    OUT = pool.map(any_job, tasks, chunksize=1)
RUNS = {r[0]: r[1:] for kind, r in OUT if kind == "main"}
VAR = {(r[0], r[1]): r[2:] for kind, r in OUT if kind == "var"}
print(f"decoding: {(time.time() - t) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Fits: 4 subsets on steps of 25 against 80 subsets on steps of 10

Fit as in the decoding notebook (`hc30.fit_imse`). A fit is called degenerate, as there, when N
is below 0 or above 1e6.

In [ ]:
fit_rows = []
for i in CHOSEN:
    s4, m4, s80, m80 = RUNS[i]
    I04, N4, I04c, N4c = hc30.fit_imse(s4, m4)
    I08, N8, I08c, N8c = hc30.fit_imse(s80, m80)
    fit_rows.append(dict(i=i, mouse=info.mouse[i], neurons=info.neurons[i],
                         misaligned=bool(sel.misaligned[i]),
                         I0_4=I04, N_4=N4, N_4_ci=N4c, I0N_4=I04 * N4,
                         I0_80=I08, N_80=N8, N_80_ci=N8c, I0N_80=I08 * N8))
FIT = pd.DataFrame(fit_rows).set_index("i")
FIT["degenerate_4"] = (FIT.N_4 < 0) | (FIT.N_4 > 1e6)
FIT["degenerate_80"] = (FIT.N_80 < 0) | (FIT.N_80 > 1e6)
pd.set_option("display.width", 160)
print(FIT[["mouse", "neurons", "misaligned", "I0_4", "N_4", "N_4_ci", "I0_80", "N_80", "N_80_ci",
           "degenerate_4", "degenerate_80"]].to_string(float_format=lambda v: f"{v:.4g}"))
print(f"degenerate fits: 4 subsets {int(FIT.degenerate_4.sum())}, 80 subsets "
      f"{int(FIT.degenerate_80.sum())} of {len(FIT)}")
ok = ~FIT.degenerate_4 & ~FIT.degenerate_80
ratio = FIT.N_80[ok] / FIT.N_4[ok]
print(f"sessions with both fits non-degenerate: {int(ok.sum())}; N(80) / N(4) median "
      f"{ratio.median():.2f}, range {ratio.min():.2f} to {ratio.max():.2f}; "
      f"I0(80) / I0(4) median {(FIT.I0_80[ok] / FIT.I0_4[ok]).median():.2f}")
print("median 95% half-interval of N relative to N: 4 subsets "
      f"{(FIT.N_4_ci / FIT.N_4.abs())[ok].median():.2f}, 80 subsets "
      f"{(FIT.N_80_ci / FIT.N_80.abs())[ok].median():.2f}")

Each session's IMSE points (1/cm^2, times 1000) against its fitted curve, for both settings
(the verifier asked for a printed table of the fits).

In [ ]:
def curve(n, I0, N):
    n = np.asarray(n, float)
    return I0 * n / (1 + n / N)

for i in CHOSEN:
    s4, m4, s80, m80 = RUNS[i]
    f = FIT.loc[i]
    print(f"session {i} {info.mouse[i]} {info.session[i]} ({info.neurons[i]} neurons"
          f"{', misaligned' if f.misaligned else ''})")
    for lab, s, m, I0, N in (("4 subsets ", s4, m4, f.I0_4, f.N_4), ("80 subsets", s80, m80, f.I0_80, f.N_80)):
        fit = curve(s, I0, N)
        r2 = 1 - np.sum((m - fit) ** 2) / np.sum((m - m.mean()) ** 2)
        print(f"  {lab}: N = {N:.4g}, R^2 = {r2:.3f}")
        print("    n     ", " ".join(f"{v:6d}" for v in s))
        print("    IMSE  ", " ".join(f"{1000 * v:6.2f}" for v in m))
        print("    fit   ", " ".join(f"{1000 * v:6.2f}" for v in fit))

## Per-animal values that the three claims use

saturation-ensemble-size takes the geometric mean of N over an animal's large-data sessions
(N <= 0 left out, as in the decoding notebook) times sqrt(20) - 1; suppl11b takes the mean I0
over an animal's large-data sessions. Mouse2028 has only 2 of its 15 sessions here, so its value
is shown for that session only.

In [ ]:
def gmean(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x) & (x > 0)]
    return np.exp(np.log(x).mean())

K = np.sqrt(20) - 1
for name, sub in (("all chosen sessions", FIT), ("without misaligned sessions", FIT[~FIT.misaligned])):
    print(name)
    for m in ("Mouse2011", "Mouse2012"):
        x = sub[sub.mouse == m]
        print(f"  {m} ({len(x)} sessions): geometric-mean N, 4 subsets {gmean(x.N_4):.0f} "
              f"(5% slope at {K * gmean(x.N_4):.0f} neurons), 80 subsets {gmean(x.N_80):.0f} "
              f"(5% slope at {K * gmean(x.N_80):.0f}); mean I0, 4 subsets {x.I0_4.mean():.3g}, "
              f"80 subsets {x.I0_80.mean():.3g}")
x = FIT[FIT.mouse == "Mouse2028"]
print("Mouse2028, one session, N: 4 subsets", x.N_4.round(0).tolist(), "; 80 subsets", x.N_80.round(0).tolist())
print(f"total run time {(time.time() - T0) / 60:.1f} min")

**Result: on these 16 sessions the fits change little with the paper's 80 subsets.** On the 16
sessions, 80 subsets on steps of 10 give fewer degenerate fits (1 against 2) and narrower
intervals for N (median half-interval 0.18 of N against 0.54), but for the 13 sessions with two
non-degenerate fits N changes by a median factor of 1.09 (0.67 to 2.02) and I0 by 0.96. The fits
that make no sense are those of sessions whose events do not track position, with either setting:
sessions 83 and 84 have flat IMSE curves and give I0 = 241 and 231 with N below 1e-5 with 80
subsets, session 51 a negative N, and session 92 an N below 1. Mouse2011's mean I0 over all 11
sessions is 23.7 with 4 subsets and 42.9 with 80; without its misaligned sessions it is 6.5e-05
and 6.3e-05. Without the misaligned sessions the geometric-mean N goes from 458 to 490 (Mouse2011)
and from 268 to 231 (Mouse2012), a 5% slope at 1,590 to 1,702 and 930 to 801 neurons. The one
Mouse2028 session is a near-linear curve with either setting (N = 1.0e9 and 3.8e4). These sessions are
all of Mouse2011 and Mouse2012 and one of Mouse2028; the animals that set the bounds of the
decoding notebook's variant without misaligned sessions are Mouse2022 and Mouse2023 (below).

## Mouse2023 (added in the second revision)

In the decoding notebook's variant without the misaligned sessions and the degenerate fits,
the lowest per-animal geometric-mean N is Mouse2022's (207) and the highest Mouse2023's (716),
which set the 5% slope range of 717 to 2,487 neurons. Mouse2023 has one large-data session. Its
4-subset fit below repeats the decoding notebook's (same seed), and it is fitted again with 40
subsets on steps of 10.

In [ ]:
ext_rows = []
for i in EXTRA:
    s4, m4, sP, mP = RUNS[i]
    reps, step = plan(i)
    I04, N4, _, N4c = hc30.fit_imse(s4, m4)
    I0P, NP, _, NPc = hc30.fit_imse(sP, mP)
    ext_rows.append(dict(i=i, mouse=info.mouse[i], session=info.session[i], neurons=info.neurons[i],
                         min_trials=info.min_trials[i], misaligned=bool(sel.misaligned[i]),
                         subsets=f"{reps} on steps of {step}", I0_4=I04, N_4=N4, N_4_ci=N4c,
                         I0_more=I0P, N_more=NP, N_more_ci=NPc))
EXT = pd.DataFrame(ext_rows).set_index("i")
EXT["degenerate"] = (EXT.N_4 < 0) | (EXT.N_4 > 1e6) | (EXT.N_more < 0) | (EXT.N_more > 1e6)
print(EXT.to_string(float_format=lambda v: f"{v:.4g}"))
okx = ~EXT.degenerate
print(f"N(more subsets) / N(4 subsets): " + ", ".join(
    f"{i} {EXT.N_more[i] / EXT.N_4[i]:.2f}" for i in EXT.index[okx]))
for i in EXTRA:
    s4, m4, sP, mP = RUNS[i]
    print(f"session {i} {info.mouse[i]} {info.session[i]} ({info.neurons[i]} neurons)")
    for lab, s, m, I0, N in (("4 subsets   ", s4, m4, EXT.I0_4[i], EXT.N_4[i]),
                             (f"{plan(i)[0]} subsets  ", sP, mP, EXT.I0_more[i], EXT.N_more[i])):
        fit = curve(s, I0, N)
        r2 = 1 - np.sum((m - fit) ** 2) / np.sum((m - m.mean()) ** 2)
        print(f"  {lab}: N = {N:.4g}, R^2 = {r2:.3f}")
        print("    n     ", " ".join(f"{v:6d}" for v in s))
        print("    IMSE  ", " ".join(f"{1000 * v:6.2f}" for v in m))
        print("    fit   ", " ".join(f"{1000 * v:6.2f}" for v in fit))
for m in ("Mouse2023",):
    x = EXT[(EXT.mouse == m) & ~EXT.misaligned]
    x4 = x.N_4[(x.N_4 > 0) & (x.N_4 < 1e6)]
    xP = x.N_more[(x.N_more > 0) & (x.N_more < 1e6)]
    print(f"{m} ({len(x)} sessions; fits with N outside 0 to 1e6 left out, as in the variant): "
          f"geometric-mean N, 4 subsets {gmean(x4):.0f} (5% slope at {K * gmean(x4):.0f} neurons, "
          f"{len(x4)} fits), {x.subsets.iloc[0]} {gmean(xP):.0f} (5% slope at {K * gmean(xP):.0f}, "
          f"{len(xP)} fits)")
print(f"run time so far {(time.time() - T0) / 60:.1f} min")

**Result.** Mouse2023's session gives N = 716 with 4 subsets, the decoding notebook's value,
and 474 with 40 subsets on steps of 10 (R squared 0.965 and 0.995), a factor of 0.66. With 474
this animal's 5% slope point would be at 1,646 neurons in place of 2,487, still above 1,402 +
10%. Mouse2022, which sets the lower bound (717 neurons), is not repeated here.

## How N depends on the decoder (added in the second revision)

The decoding notebook uses libsvm's L2 linear SVM with C = 1 and voting; the paper's code uses
MATLAB fitcecoc with an unregularised fit and loss-weighted decoding. Here the 4-subset run is
repeated with C = 0.1, C = 10, and loss-weighted decoding with the hinge loss at C = 1
(`hc30.OvOLinearSVM`), on the 9 sessions listed in the data check. Each is compared with the
same session's 4-subset fit at C = 1 with voting, and, to show how much N moves between two
runs of the same decoder, with its fit to more subsets.

In [ ]:
vrows = []
for i in VAR_SESSIONS:
    base = FIT.loc[i] if i in FIT.index else EXT.loc[i].rename({"N_more": "N_80", "I0_more": "I0_80"})
    row = dict(i=i, mouse=info.mouse[i], N_vote_C1=base.N_4, N_more_subsets=base.N_80)
    for k, (lab, _, _) in enumerate(VARIANTS):
        s, m = VAR[(i, k)]
        I0, N, _, _ = hc30.fit_imse(s, m)
        row[f"N {lab}"] = N
        row[f"IMSE all {lab}"] = m[-1]
    row["IMSE all vote C1"] = RUNS[i][1][-1]
    vrows.append(row)
V = pd.DataFrame(vrows).set_index("i")
print(V.to_string(float_format=lambda v: f"{v:.4g}"))
base_ok = (V.N_vote_C1 > 0) & (V.N_vote_C1 < 1e6)
for lab, _, _ in VARIANTS:
    n = V[f"N {lab}"]
    good = (n > 0) & (n < 1e6) & base_ok
    r = n[good] / V.N_vote_C1[good]
    ri = V[f"IMSE all {lab}"] / V["IMSE all vote C1"]
    print(f"{lab}: degenerate fits {int(((n <= 0) | (n >= 1e6)).sum())} of {len(V)}; N / N(C = 1, voting) median "
          f"{r.median():.2f}, range {r.min():.2f} to {r.max():.2f}; IMSE at all neurons / C = 1 "
          f"voting median {ri.median():.2f}")
okm = base_ok & (V.N_more_subsets > 0) & (V.N_more_subsets < 1e6)
r = V.N_more_subsets[okm] / V.N_vote_C1[okm]
print(f"for comparison, more subsets against 4 at C = 1 with voting: median {r.median():.2f}, "
      f"range {r.min():.2f} to {r.max():.2f}")
for m in ("Mouse2011", "Mouse2012"):
    x = V[V.mouse == m]  # Mouse2011: the five sessions with 250 neurons or fewer
    print(f"{m} ({len(x)} sessions), geometric-mean N: C = 1 voting {gmean(x.N_vote_C1):.0f}, "
          + ", ".join(f"{lab} {gmean(x[f'N {lab}']):.0f}" for lab, _, _ in VARIANTS))
print(f"total run time {(time.time() - T0) / 60:.1f} min")

**Result: N depends on the decoder by about as much as on the number of subsets, and the
paper's loss-weighted decoding lowers it.** On the 9 sessions, N with C = 0.1 is a median 1.25
times N with C = 1 (0.58 to 1.75), with C = 10 0.97 times (0.41 to 1.71), and with
loss-weighted decoding 0.71 times (0.29 to 1.81); fitting the same decoder to more subsets moves
N by a median factor of 0.82 (0.66 to 1.46). The geometric-mean N of Mouse2011's five sessions
is 523 with C = 1 and voting, and 583, 503 and 369 with the three variants; of Mouse2012's
three sessions 268, and 278, 295 and 208. Loss-weighted decoding also lowers the IMSE with all
neurons (median factor 0.88). The saturation range was not recomputed with loss-weighted
decoding, which needs the whole decoding run.

## Summary

**Result.**

| Claim | Outcome |
|---|---|
| saturation-ensemble-size | tested in the decoding notebook; this notebook checks its fit |
| suppl11b-i0-vs-single-cell-snr | tested in the decoding notebook; this notebook checks its fit |
| suppl8-task-performance-and-accuracy | tested in the decoding notebook; this notebook checks its fit |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

With the paper's 80 subsets on steps of 10, N changes by a median factor of 1.09 on the 13
sessions of Mouse2011, Mouse2012 and Mouse2028 with two non-degenerate fits, and the failed fits
are those of misaligned sessions with either setting. Mouse2023's N falls from 716 to 474 with
40 subsets; Mouse2022 was not repeated. On 9 sessions the paper's loss-weighted decoding gives
a median 0.71 of the N of this attempt's voting decoder. The fits on which saturation-ensemble-size
and suppl8 rest therefore depend on the decoder, and the effect of the full method on those
claims is not known.